In [2]:
import os
import re
from datetime import datetime
import matplotlib.pyplot as plt
import sunpy.map
import image_registration
from astropy.io import fits
from matplotlib.colors import LogNorm, Normalize
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import itertools
from PIL import Image
from IPython.display import HTML
import time
%matplotlib notebook

In [3]:
file_path = "E:/python/projects/alfven/data/transform_data/"
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]
def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

files = sorted(files, key=extract_datetime)
file_fsi = "E:/python/projects/alfven/data/solo_L2_eui-fsi174-image_20220330T042045225_V02.fits"

## hri图像之间进行对齐

In [11]:
def sub_data(euv, x_range, y_range):
    """
    选取一定像素范围图像进行比对
    :param euv: fits文件或sunpy.map.Map对象
    :param x_range: x轴像素范围
    :param y_range: x轴像素范围
    :return: 裁剪后的数据
    """
    if type(euv) is str:
        euv_data = fits.getdata(euv)
    else:
        euv_data = euv.data
    
    return euv_data[y_range[0]:y_range[1], x_range[0]:x_range[1]]

def get_offset(euv1, euv2, x_range, y_range):
    """
    得到hri图像之间的偏移量
    :param euv1: 标定图像，将会被投影到euv2的观察者系，fits文件或sunpy.map.Map对象
    :param euv2: 需要被移动的图像，fits文件或sunpy.map.Map对象
    :param x_range: x轴像素范围
    :param y_range: x轴像素范围
    :return: 平移量
    """
    if type(euv1) is str:
        euv1 = sunpy.map.Map(euv1)
    if type(euv2) is str:
        euv2 = sunpy.map.Map(euv2)
    
    euv1_reproject = euv1.reproject_to(euv2.wcs)
    data1 = sub_data(euv1_reproject, x_range, y_range)
    data2 = sub_data(euv2, x_range, y_range)
    dx, dy = image_registration.chi2_shift(data2, data1, return_error=False)
    
    return dx, dy

def offset_map(euv, dx, dy):
    """
    平移图像
    :param euv: fits文件或sunpy.map.Map对象
    :param dx: x方向平移像素值
    :param dy: y方向平移像素值
    :return: 平移后的图像，sunpy.map.Map对象
    """
    if type(euv) is str:
        euv = sunpy.map.Map(euv)
        
    data = euv.data
    offset_data = image_registration.fft_tools.shift2d(data, dx, dy)
    offset_data[offset_data < np.nanmin(data)] = np.nan
    
    return sunpy.map.Map(offset_data, euv.meta)

def euv_plot(euv, return_fig=False, **kwargs):
    if type(euv) is str:
        euv = sunpy.map.Map(euv)
    fig = plt.figure()
    ax = fig.add_subplot(projection=euv)
    euv.plot(axes=ax, **kwargs)
    if return_fig:
        return fig

## 选取一定像素范围进行比较

In [ ]:
hri = sunpy.map.Map(files[0])
euv_plot(hri, cmap='sdoaia171', norm=Normalize())

选取如下坐标范围：x方向-400至100arcsec，y方向-2800至-2300arcsec

In [ ]:
bottom_left = SkyCoord(-400*u.arcsec, -2800*u.arcsec, frame=hri.coordinate_frame)
top_right = SkyCoord(100*u.arcsec, -2300*u.arcsec, frame=hri.coordinate_frame)

sub_map = hri.submap(bottom_left, top_right=top_right)
euv_plot(sub_map, cmap='sdoaia171', norm=Normalize())

计算坐标范围：

In [ ]:
crpix1 = hri.fits_header['crpix1']
crpix2 = hri.fits_header['crpix2']
crval1 = hri.fits_header['crval1']
crval2 = hri.fits_header['crval2']
cdelt1 = hri.fits_header['cdelt1']
cdelt2 = hri.fits_header['cdelt2']

left = crpix1 + (-400 - crval1)/cdelt1
right = crpix1 + (100 - crval1)/cdelt1
bottom = crpix2 + (-2800 - crval2)/cdelt2
top = crpix2 + (-2300 - crval2)/cdelt2

print(f'left:{left}, right:{right}, bottom:{bottom}, top:{top}')

将像素范围取为整数，并保持正方形：

In [ ]:
x_range = (640, 1660)
y_range = (1250, 2270)

In [ ]:
example = offset_map(hri, 10, 10)
euv_plot(example, cmap='sdoaia171', norm=Normalize())

得到hri图像之间的偏移量

In [ ]:
offset_vect = []
for i in tqdm(range(600), total=600, desc='computing offsets'):
    dx, dy = get_offset(files[0], files[i], x_range, y_range)
    offset_vect.append([dx, dy])

## hri图像与fsi图像进行对齐：

In [ ]:
def l1_residual_sum(hri, fsi, offset, x_range, y_range):
    hri = offset_map(hri, offset[0], offset[1])
    if type(fsi) is str:
        fsi = sunpy.map.Map(fsi)
    
    data_hri = sub_data(hri, x_range, y_range)
    data_fsi = sub_data(fsi, x_range, y_range)
    
    def normalize(image):
        return (image - np.mean(image)) / np.std(image)
    
    return np.nansum(np.abs(normalize(data_hri)-normalize(data_fsi)))

def iterate_alignment(hri, fsi, x_range, y_range, accuracy, offset_range):
    if type(fsi) is str:
        fsi = sunpy.map.Map(fsi)
    if type(hri) is str:
        hri = sunpy.map.Map(hri)
    fsi = fsi.reproject_to(hri.wcs)
    accuracy = np.array(accuracy)
    range_x, range_y = offset_range[0], offset_range[1]
    min_diff = np.inf
    best_offset = (0, 0)
    for a in accuracy:
        offset_vect_x = np.arange(range_x[0], range_x[1]+a, a)
        offset_vect_y = np.arange(range_y[0], range_y[1]+a, a)
        offsets = list(itertools.product(offset_vect_x, offset_vect_y))
        
        with ThreadPoolExecutor() as executor:
            futures = {executor.submit(l1_residual_sum, hri, fsi, offset, x_range, y_range): offset for offset in offsets}
            for future in tqdm(as_completed(futures), total=len(offsets), desc=f"computing best offset, accuracy: {a}", leave=False):
                diff = future.result()
                if diff < min_diff:
                    min_diff = diff
                    best_offset = futures[future]
                    
        range_x = (best_offset[0]-a, best_offset[0]+a)
        range_y = (best_offset[1]-a, best_offset[1]+a)
    
    return best_offset

def compare(hri, fsi, offset=None):
    if offset is not None:
        hri = offset_map(hri, offset[0], offset[1])
    else:
        if type(hri) is str:
            hri = sunpy.map.Map(hri)
    if type(fsi) is str:
        fsi = sunpy.map.Map(fsi)
    fsi = fsi.reproject_to(hri.wcs)
    
    def get_submap(hri):
        bottom_left = SkyCoord(-400*u.arcsec, -2800*u.arcsec, frame=hri.coordinate_frame)
        top_right = SkyCoord(100*u.arcsec, -2300*u.arcsec, frame=hri.coordinate_frame)
        sub_map = hri.submap(bottom_left, top_right=top_right)
        return sub_map
    
    hri = get_submap(hri)
    fsi = get_submap(fsi)
        
    with plt.ioff():
        fig = plt.figure()
        ax = plt.subplot(projection=fsi)
        fsi.plot(axes=ax, norm=Normalize(), cmap='sdoaia171')
        fig.savefig('./fig/fsi.png', dpi=300)
        ax.clear()

        ax = fig.add_subplot(projection=hri)
        hri.plot(axes=ax, norm=Normalize(), cmap='sdoaia171')
        fig.savefig('./fig/hri.png', dpi=300)
    
        image_files = ['./fig/fsi.png', './fig/hri.png']

        images = [Image.open(image) for image in image_files]

        images[0].save('./fig/fsi_vs_hri.gif', save_all=True, append_images=images[1:], duration=500, loop=0)
        
    current_directory = os.getcwd()
    relative_path = os.path.relpath('./fig/fsi_vs_hri.gif', current_directory)
    timestamp = int(time.time())
    
    return HTML(f'<img src="{relative_path}?{timestamp}" style="height:480px;">')

进行初步比对，发现像素偏移量为(9, 2)pix时效果不错，设定像素查找范围为((7, 11), (0, 4))

In [ ]:
compare(files[0], file_fsi, (10, 2))

In [ ]:
best_x, best_y = iterate_alignment(files[0], file_fsi, (640, 1660), (1250, 2270), [1, 0.25, 0.05, 0.01], ((7, 11), (0, 4)))
best_x = round(best_x, 2)
best_y = round(best_y, 2)
print(f'best offset: offset pixels in x: {best_x}, y: {best_y}')

查看结果：

In [ ]:
compare(files[0], file_fsi, (10.32, 2.12))

## 将hri图像进行对齐

In [5]:
file0_to_fsi = (10.32, 2.12)
hri_to_file0 = np.load('./data/offset_array.npy')

In [10]:
def alignment_maps(euv, dx, dy, output_path):
    offseted_map = offset_map(euv, dx, dy)
    file_name = offseted_map.fits_header['filename']
    offseted_map.save(os.path.join(output_path, file_name), filetype='fits', overwrite=True)

In [12]:
alignment_path = './data/hrieuvopn/alignment_data/'
for i, file in enumerate(tqdm(files, total=len(files), desc='offsetting files')):
    dx = hri_to_file0[i][0] + file0_to_fsi[0]
    dy = hri_to_file0[i][1] + file0_to_fsi[1]
    alignment_maps(file, dx, dy, alignment_path)

offsetting files:   0%|          | 0/600 [00:00<?, ?it/s]

## 进行滑动平均

In [13]:
alignment_file_path = "E:/python/projects/alfven/data/hrieuvopn/alignment_data/"
transform_files = [os.path.abspath(os.path.join(alignment_file_path, file)) for file in os.listdir(alignment_file_path) if file.endswith('.fits')]
def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

transform_files = sorted(transform_files, key=extract_datetime)

In [14]:
from queue import Queue

window_size = 10
mid = window_size // 2
q_data = Queue()
head = 0
sum_data = np.zeros_like(sunpy.map.Map(transform_files[0]).data)
save_path = "E:/python/projects/alfven/data/hrieuvopn/alignment_data/10moving/"

for point in tqdm(range(len(transform_files)+1), desc='moving average', total=len(files)+1):
    if q_data.qsize() == window_size:  # 队列中有window_size个元素，进行计算平均值操作
        name = os.path.basename(transform_files[head+mid])  # 得到保存的文件名
        mean_data = sum_data / window_size # 平均数据
        _, header = fits.getdata(transform_files[head+mid], header=True) # 获取头信息
        mean_map = sunpy.map.Map(mean_data, header)
        mean_map.save(save_path + name, filetype='fits', overwrite=True) # 制作并储存平均后的文件
        
        if point<len(transform_files):
            sum_data -= q_data.get()  # 出队并从总数据减去队头数据
            data_in = fits.getdata(transform_files[point])
            sum_data += data_in
            q_data.put(data_in)
            head += 1

    elif q_data.qsize() < window_size:
        data_in = fits.getdata(transform_files[point])
        q_data.put(data_in)
        sum_data += data_in  # 增加初始化时的数据累加
         
    else:
        raise RuntimeError("Something wrong in the algorithm.")

moving average:   0%|          | 0/601 [00:00<?, ?it/s]